# Verify Risk of Bias (RoB) References

This notebook verifies that the "Smart Search" logic is correctly identifying 
foundational Risk of Bias literature (e.g., Higgins 2011, Sterne 2019).

In [3]:
import pandas as pd
from IPython.display import display, HTML
from scraper.export_reader import load_latest_export

# Load the latest export
reader = load_latest_export()

Loading latest export: processed_export_1780342516.json


## Detected RoB References

Below are all the references across all papers that the system has flagged as an "RoB Tool" signature.

In [4]:
rob_refs = reader.identify_rob_references()

if rob_refs.empty:
    display(HTML("<div style='color: red;'><b>No RoB references identified.</b></div>"))
else:
    print(f"Found {len(rob_refs)} RoB-related references.\n")
    
    # Sort for easier reading
    rob_refs = rob_refs.sort_values(['paper_id', 'ref_id'])
    
    # Get all sections to find where they are cited
    sections_df = reader.get_all_sections_dataframe()
    
    for idx, ref in rob_refs.iterrows():
        pid = ref['paper_id']
        rid = ref['ref_id']
        
        # Find sections citing this
        mask = sections_df.apply(lambda x: x['paper_id'] == pid and rid in x['citations'], axis=1)
        citing_sections = sections_df[mask]['heading'].tolist()
        sections_str = ", ".join(citing_sections) if citing_sections else "(None detected in sections)"

        html = f"""
        <div style="border: 1px solid #ccc; padding: 15px; border-radius: 8px; margin-bottom: 20px; background-color: #f9f9f9;">
            <div style="display: flex; justify-content: space-between;">
                <b style="font-size: 1.1em; color: #2c3e50;">Paper: {pid} | Ref: {rid}</b>
                <span style="color: #7f8c8d; font-family: monospace;">DOI: {ref['doi'] or 'N/A'}</span>
            </div>
            <hr style="margin: 10px 0;">
            <div style="margin-bottom: 8px;"><b>Title:</b> {ref['title'] or 'N/A'}</div>
            <div style="margin-bottom: 8px;"><b>Full Text:</b> <i style="color: #34495e;">{ref['text']}</i></div>
            <div style="background-color: #e8f6f3; padding: 5px 10px; border-radius: 4px; font-size: 0.9em;">
                <b>Cited in Sections:</b> {sections_str}
            </div>
        </div>
        """
        display(HTML(html))

Found 98 RoB-related references.

